In [1]:
import time
from pathlib import Path

import torch
import pandas as pd

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import sacrebleu

In [2]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(device)


if torch.cuda.is_available():
    print(
        torch.cuda.get_device_name(0)
    )

cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
MODEL_NAME = (
    "alirezamsh/small100"
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)


model.to(device)

model.eval()


print("loaded")

Loading weights:   0%|          | 0/275 [00:00<?, ?it/s]

loaded


In [4]:
print(
    list(
        tokenizer.lang_code_to_token.keys()
    )[:30]
)

['af', 'am', 'ar', 'ast', 'az', 'ba', 'be', 'bg', 'bn', 'br', 'bs', 'ca', 'ceb', 'cs', 'cy', 'da', 'de', 'el', 'en', 'es', 'et', 'fa', 'ff', 'fi', 'fr', 'fy', 'ga', 'gd', 'gl', 'gu']


In [5]:
def translate(
    text,
    src_lang,
    tgt_lang
):

    tokenizer.src_lang = src_lang


    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(device)



    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            forced_bos_token_id =
            tokenizer.get_lang_id(
                tgt_lang
            ),

            max_length=128,

            num_beams=5
        )


    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


    return result

In [6]:
print(
    translate(
        "I am a student.",
        "en",
        "uz"
    )
)

: I am a student.


In [7]:
DATA_PATH = Path(
    r"D:\dev\projects\fourlang_translation\data\clean\en_uz\tatoeba_en_uz_latin.csv"
)


df = pd.read_csv(
    DATA_PATH
)


print(df.shape)

df.head()

(429, 5)


,en_id,en,uz_id,uz,uz_has_cyrillic
0,16996,Mind your own business!,423846,Ishingizni qiling!,False
1,16996,Mind your own business!,423847,Ishingni qil!,False
2,16996,Mind your own business!,423848,Ishinglarni qilinglar!,False
3,19904,The customer did not come.,2790055,Xaridor kelmadi.,False
4,20392,Never mind.,423857,Hech gap yo'q.,False


In [8]:
test_df = df.sample(
    100,
    random_state=42
)


test_df.head()

,en_id,en,uz_id,uz,uz_has_cyrillic
421,2362724,Good night.,13758141,Xayrli tun.,False
75,2499490,Where is Laurie?,2499491,Laurie qayerda?,False
177,64956,I was hungry.,3834516,Men och edim.,False
30,410907,Where are my watches?,423842,Mening soatlarim qani?,False
360,5293174,"I refuse to use public restrooms, as they are ...",11282118,Juda antigigiyenik bo‘lganidan jamoat hojatxon...,False


In [9]:
results_en_uz=[]


for _,row in tqdm(
    test_df.iterrows(),
    total=len(test_df)
):


    start=time.time()


    pred=translate(
        row["en"],
        "en",
        "uz"
    )


    latency=time.time()-start


    results_en_uz.append(
        {
            "src_lang":"en",
            "tgt_lang":"uz",

            "source":
                row["en"],

            "reference":
                row["uz"],

            "prediction":
                pred,

            "latency":
                latency
        }
    )



en_uz_result = pd.DataFrame(
    results_en_uz
)


en_uz_result.head()

  0%|          | 0/100 [00:00<?, ?it/s]

,src_lang,tgt_lang,source,reference,prediction,latency
0,en,uz,Good night.,Xayrli tun.,al good night.,0.249095
1,en,uz,Where is Laurie?,Laurie qayerda?,: Where is Laurie?,0.045957
2,en,uz,I was hungry.,Men och edim.,was hungry.,0.067818
3,en,uz,Where are my watches?,Mening soatlarim qani?,: Where are my watches?,0.077741
4,en,uz,"I refuse to use public restrooms, as they are ...",Juda antigigiyenik bo‘lganidan jamoat hojatxon...,"to use public restrooms, as they are very unhy...",0.123693


In [10]:
results_uz_en=[]


for _,row in tqdm(
    test_df.iterrows(),
    total=len(test_df)
):


    start=time.time()


    pred=translate(
        row["uz"],
        "uz",
        "en"
    )


    latency=time.time()-start



    results_uz_en.append(
        {
            "src_lang":"uz",
            "tgt_lang":"en",

            "source":
                row["uz"],

            "reference":
                row["en"],

            "prediction":
                pred,

            "latency":
                latency
        }
    )


uz_en_result=pd.DataFrame(
    results_uz_en
)


uz_en_result.head()

  0%|          | 0/100 [00:00<?, ?it/s]

,src_lang,tgt_lang,source,reference,prediction,latency
0,uz,en,Xayrli tun.,Good night.,idagi tun.,0.041689
1,uz,en,Laurie qayerda?,Where is Laurie?,idagi Laurie?,0.039557
2,uz,en,Men och edim.,I was hungry.,idagi edim.,0.044426
3,uz,en,Mening soatlarim qani?,Where are my watches?,vlarim qani?,0.051666
4,uz,en,Juda antigigiyenik bo‘lganidan jamoat hojatxon...,"I refuse to use public restrooms, as they are ...",idagi juda antikigiyenik bo‘liqidan qo‘yot hoj...,0.171640


In [11]:
bleu_en_uz = sacrebleu.corpus_bleu(
    en_uz_result["prediction"].tolist(),

    [
        en_uz_result["reference"].tolist()
    ]
)


print(
    "en→uz BLEU:",
    bleu_en_uz.score
)

en→uz BLEU: 0.24513318035331913


In [12]:
bleu_uz_en = sacrebleu.corpus_bleu(
    uz_en_result["prediction"].tolist(),

    [
        uz_en_result["reference"].tolist()
    ]
)


print(
    "uz→en BLEU:",
    bleu_uz_en.score
)

uz→en BLEU: 0.5158710993966064


In [13]:
chrf_en_uz = sacrebleu.corpus_chrf(
    en_uz_result["prediction"].tolist(),

    [
        en_uz_result["reference"].tolist()
    ]
)


print(
    chrf_en_uz.score
)

9.034570520970934


In [14]:
chrf_uz_en = sacrebleu.corpus_chrf(
    uz_en_result["prediction"].tolist(),

    [
        uz_en_result["reference"].tolist()
    ]
)


print(
    chrf_uz_en.score
)

9.117027350182909


In [15]:
print(
    "en→uz latency"
)


print(
    en_uz_result["latency"]
    .describe()
)



print(
    "uz→en latency"
)


print(
    uz_en_result["latency"]
    .describe()
)

en→uz latency
count    100.000000
mean       0.056457
std        0.031053
min        0.027155
25%        0.044364
50%        0.050764
75%        0.058667
max        0.249095
Name: latency, dtype: float64
uz→en latency
count    100.000000
mean       0.057008
std        0.021590
min        0.030721
25%        0.041804
50%        0.052938
75%        0.064761
max        0.171640
Name: latency, dtype: float64


In [17]:
RESULT_DIR = Path(
    r"D:\dev\projects\fourlang_translation\results"
)


RESULT_DIR.mkdir(
    exist_ok=True
)


en_uz_result.to_csv(
    RESULT_DIR/
    "small100_baseline_en_uz.csv",
    index=False,
    encoding="utf-8-sig"
)


uz_en_result.to_csv(
    RESULT_DIR/
    "small100_baseline_uz_en.csv",
    index=False,
    encoding="utf-8-sig"
)